# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library. All dataset components are referenced by their Croissant `@id` fields for reproducibility and transparency.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Below we enumerate all record sets, displaying their `@id`, name, and contained field `@id`s.

In [ ]:
# Enumerate all record sets and their fields
record_sets = list(dataset.record_sets)
print("Available record sets:\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} (Name: {rs.get('name', 'N/A')})")
    fields = rs.get('field', [])
    if fields and isinstance(fields, list):
        for f in fields:
            field_id = f['@id'] if isinstance(f, dict) and '@id' in f else f
            print(f"  Field: {field_id}")
    elif fields:
        # Case: single field or string
        print(f"  Field: {fields if isinstance(fields, str) else fields.get('@id', '(unknown)')}")
    print()
# For demonstration, show the first few records from each record set
for rs in record_sets:
    print(f"Sample records for record set {rs['@id']}:")
    try:
        for idx, rec in enumerate(dataset.records(record_set=rs['@id'])):
            print(rec)
            if idx >= 1:
                break
    except Exception as e:
        print(f"  Could not load records: {e}")
    print()


## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames using their `@id`s. Replace `<record_set_id>` below with actual IDs from the overview.

In [ ]:
# Extract data from each record set
# Gather record set IDs
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(f"Sample rows for {record_set_id}:")
        print(df.head())
    except Exception as e:
        print(f"Could not extract records for {record_set_id}: {e}")

# For demonstration, select the first record set if available
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"Selected record set for further analysis: {selected_record_set_id}")
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping.
- All operations reference columns/fields by their `@id` values as shown above.

Below, we demonstrate filtering and normalization using the first numeric field found in the selected record set.

In [ ]:
if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    # Find a numeric field using column dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Use @id as column name
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical (non-numeric) field by @id
        group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields available for EDA.")
else:
    print("No record set selected or DataFrame unavailable.")

## 5. Visualization
Visualize distributions or relationships between fields using column `@id`s. For numeric fields, we show a histogram and a boxplot if relevant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and selected_record_set_id in dataframes and numeric_fields:
    field = numeric_field_id
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {field} (by @id)")
    plt.xlabel(field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by a categorical field
    if group_fields:
        group_field_id = group_fields[0]
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[field])
        plt.title(f"{field} by {group_field_id} (by @id)")
        plt.xlabel(group_field_id)
        plt.ylabel(field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields or groupable categorical fields for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load the FAIR^2 dataset from its Croissant schema URL, referenced all entities by their `@id` fields, and performed basic exploratory data analysis and visualization. Key findings can be further summarized here based on the fields and distributions observed.

- All operations used reproducible references to the Croissant schema via `@id`s.
- The dataset's structure, available fields, and example data were explored and transformed.
- Further analysis can build upon these DataFrames using domain knowledge and field documentation from the Croissant metadata.

For further insights, consult the dataset documentation and explore more advanced modeling or statistical techniques.